# 03 — Feature Engineering

SQL-first aggregations (via DuckDB) across all supplementary tables,
merged into the application base table with domain-driven derived features,
missing value indicators, and Bayesian smoothed target encoding.

**Pipeline:** Raw CSVs (8 tables) -> DuckDB SQL aggs -> Derived features -> Missing indicators -> Target encoding -> ~300 columns

In [7]:
import sys
sys.path.insert(0, "../src")

import pathlib
import pandas as pd
import numpy as np
import duckdb

from feature_engineering import (
    load_tables, run_sql_file, add_derived_features,
    add_missing_indicators, target_encode_categoricals,
)

RAW = pathlib.Path("../data/raw")
SQL = pathlib.Path("../sql")
OUT = pathlib.Path("../data/processed")


In [8]:
con = duckdb.connect()
load_tables(con, RAW)

  Loaded application_test (48,744 rows)
  Loaded application_train (307,511 rows)
  Loaded bureau (1,716,428 rows)
  Loaded bureau_balance (27,299,925 rows)
  Loaded credit_card_balance (3,840,312 rows)
  Loaded installments_payments (13,605,401 rows)
  Loaded POS_CASH_balance (10,001,358 rows)
  Loaded previous_application (1,670,214 rows)


## Run SQL Aggregations

In [9]:
agg_frames = {}
for sql_file in sorted(SQL.glob("*.sql")):
    name = sql_file.stem
    agg_frames[name] = run_sql_file(con, sql_file)
    print(f"{name}: {agg_frames[name].shape}")

print(f"\n{len(agg_frames)} SQL aggregation tables built")


bureau_agg: (305811, 28)
bureau_balance_agg: (134542, 14)
credit_card_agg: (92447, 24)
installment_features: (336935, 22)
pos_cash_agg: (334359, 20)
previous_app_agg: (338857, 23)

6 SQL aggregation tables built


## Merge to Application Base

In [10]:
app = con.execute("SELECT * FROM application_train").fetchdf()
print(f"Base: {app.shape}")

for name, agg_df in agg_frames.items():
    if "SK_ID_CURR" in agg_df.columns:
        app = app.merge(agg_df, on="SK_ID_CURR", how="left")
        print(f"+ {name}: now {app.shape[1]} cols")

app = add_derived_features(app)
print(f"\nAfter derived features: {app.shape}")

app = add_missing_indicators(app)
print(f"After missing indicators: {app.shape}")

app = target_encode_categoricals(app, target_col="TARGET")
print(f"After target encoding: {app.shape}")

print(f"\nFinal shape: {app.shape[0]:,} rows x {app.shape[1]} cols")
print(f"Default rate: {app['TARGET'].mean():.2%}")


Base: (307511, 122)
+ bureau_agg: now 149 cols
+ bureau_balance_agg: now 162 cols
+ credit_card_agg: now 185 cols
+ installment_features: now 206 cols
+ pos_cash_agg: now 225 cols
+ previous_app_agg: now 247 cols

After derived features: (307511, 274)
After missing indicators: (307511, 286)
After target encoding: (307511, 302)

Final shape: 307,511 rows x 302 cols
Default rate: 8.07%


In [11]:
OUT.mkdir(parents=True, exist_ok=True)
app.to_csv(OUT / "train_features.csv", index=False)
print(f"Saved train_features.csv ({app.shape[0]:,} rows x {app.shape[1]} cols)")
con.close()


Saved train_features.csv (307,511 rows x 302 cols)


## Feature Summary


In [12]:
# Feature categories
exclude = ["SK_ID_CURR", "SK_ID_PREV", "TARGET"]
feature_cols = [c for c in app.columns if c not in exclude]

n_numeric = app[feature_cols].select_dtypes(include="number").shape[1]
n_missing_flags = len([c for c in feature_cols if c.startswith("MISS_")])
n_te = len([c for c in feature_cols if c.endswith("_TE")])
n_derived = len([c for c in feature_cols if c in [
    "LOAN_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_GOODS_RATIO",
    "EMPLOYMENT_YEARS", "AGE_YEARS", "EMPLOYMENT_AGE_RATIO",
    "EXT_SOURCE_MEAN", "EXT_SOURCE_MAX", "EXT_SOURCE_MIN",
    "EXT_SRC_2x3", "EXT_SRC_1x2", "EXT_SRC_1x3", "EXT_SRC_PRODUCT",
    "DOC_COUNT", "BUREAU_ENQUIRY_TOTAL", "PAYMENT_RATE", "CREDIT_OVERCHARGE",
    "ANNUITY_LENGTH_YEARS", "INCOME_PER_FAMILY", "TIME_EMPLOYED_RATIO",
]])

print(f"Total features: {len(feature_cols)}")
print(f"  Numeric features: {n_numeric}")
print(f"  Missing indicators: {n_missing_flags}")
print(f"  Target-encoded categoricals: {n_te}")
print(f"  Key derived features: {n_derived}")
print(f"\nMissing values: {app[feature_cols].isna().sum().sum():,} total NaN cells")
print(f"Columns with >50% NaN: {(app[feature_cols].isna().mean() > 0.5).sum()}")


Total features: 300
  Numeric features: 283
  Missing indicators: 12
  Target-encoded categoricals: 0
  Key derived features: 14

Missing values: 20,823,577 total NaN cells
Columns with >50% NaN: 80
